In [1]:
!pip install transformers datasets torch scikit-learn

In [2]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
#Testing the tokenizer
tokenizer("Hello! Tesiting this...")

{'input_ids': [101, 7592, 999, 8915, 28032, 2075, 2023, 1012, 1012, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
#Tokenizing the dataset
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [6]:
#Loading BERT model for classification (pretrained BERT + classification layer)
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
#Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(10000)),
eval_dataset=tokenized_dataset["test"].shuffle(seed=42).select(range(2000))
)

trainer.train()

Step,Training Loss
50,1.164677
100,0.539926
150,0.409884
200,0.401381
250,0.407256
300,0.462155
350,0.384399
400,0.412858
450,0.420936
500,0.340253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=0.3089865665435791, metrics={'train_runtime': 585.0897, 'train_samples_per_second': 34.183, 'train_steps_per_second': 4.273, 'total_flos': 1315578900480000.0, 'train_loss': 0.3089865665435791, 'epoch': 2.0})

In [22]:
import torch
import torch.nn.functional as F
import re

labels = ["World", "Sports", "Business", "Sci/Tech"]

def is_meaningful_text(text):
    # Reject if no real words
    words = text.split()

    if len(words) < 3:
        return False

    # Reject if no alphabetic words of reasonable length
    valid_words = [w for w in words if re.search("[a-zA-Z]{3,}", w)]

    return len(valid_words) >= 2


def predict(text, threshold=0.75):

    model.eval()

    # Step 1: reject meaningless input
    if not is_meaningful_text(text):
        return "Invalid input (not meaningful text)"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

    # Move input tensors to the same device as the model
      inputs = {k: v.to(model.device) for k, v in inputs.items()}
      outputs = model(**inputs)

    logits = outputs.logits

    probs = F.softmax(logits, dim=1)

    confidence, predicted_class_id = torch.max(probs, dim=1)

    confidence = confidence.item()
    predicted_class_id = predicted_class_id.item()

    # Step 2: reject low confidence predictions
    if confidence < threshold:
        return f"Uncertain prediction (confidence: {confidence:.2f})"

    return f"{labels[predicted_class_id]} (confidence: {confidence:.2f})"

In [23]:
#Testing
text = "Apple released a new iPhone with advanced technology"

prediction = predict(text)

print("Text:", text)
print("Predicted category:", prediction)

Text: Apple released a new iPhone with advanced technology
Predicted category: Sci/Tech (confidence: 0.99)


In [37]:
#Tesing on random custom lines
examples = [
    "The football team won the championship",
    "Stock markets reached record highs",
    "Scientists discovered a new planet",
    "The president met with foreign leaders"
]

for text in examples:
    print("Text:", text)
    print("Prediction:", predict(text))
    print()

Text: The football team won the championship
Prediction: Sports (confidence: 0.97)

Text: Stock markets reached record highs
Prediction: Business (confidence: 0.99)

Text: Scientists discovered a new planet
Prediction: Sci/Tech (confidence: 0.99)

Text: The president met with foreign leaders
Prediction: World (confidence: 1.00)



In [38]:
#User input for prediction
user_input = input("Enter your text: ")

result = predict(user_input)

print("\nResult:", result)

Enter your text: French President is visiting India

Result: World (confidence: 1.00)


In [33]:
#Saving the model
model.save_pretrained("bert_text_classifier")
tokenizer.save_pretrained("bert_text_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_text_classifier/tokenizer_config.json',
 'bert_text_classifier/tokenizer.json')